# Agent IA d'analyse de fraude — notebook de démonstration

Ce notebook montre chaque brique de l'agent **sans passer par le LLM** : la base SQL, le scoring par l'API du projet 2, le RAG sur les procédures, le serveur MCP, les skills, les garde-fous et la validation humaine. Il se termine par les résultats de l'évaluation.

Prérequis : l'API de fraude du projet 2 tourne sur `FRAUD_API_URL` (voir README).

In [1]:
import os, shutil, sys, tempfile, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault('RAG_EMBEDDINGS', 'local')  # 'gemini' si une clé est configurée
from app import config
# On travaille sur une copie de la base pour ne pas la modifier
tmp = Path(tempfile.mkdtemp()); shutil.copy(config.ROOT/'data'/'bank.db', tmp/'bank.db')
config.BANK_DB_PATH = tmp/'bank.db'; config.CHROMA_DIR = tmp/'chroma'
import pandas as pd, sqlite3
con = sqlite3.connect(config.BANK_DB_PATH)

## 1. La base de données d'entreprise (SQLite)
20 000 transactions issues du jeu de données du projet 2 (toutes les 492 fraudes + 19 508 transactions normales), réparties sur 300 cartes, avec des colonnes métier synthétiques.

In [2]:
pd.read_sql('SELECT (SELECT COUNT(*) FROM cards) AS cartes, COUNT(*) AS transactions, SUM(label_is_fraud) AS fraudes FROM transactions', con)

,cartes,transactions,fraudes
0,300,20000,492


In [3]:
pd.read_sql('''SELECT merchant_category, COUNT(*) AS n, ROUND(100.0*AVG(label_is_fraud),2) AS taux_fraude_pct
FROM transactions GROUP BY 1 ORDER BY 3 DESC''', con)

,merchant_category,n,taux_fraude_pct
0,transfert d'argent,159,40.25
1,jeux en ligne,152,40.13
2,bijouterie,164,35.98
3,électronique,143,33.57
4,crypto-monnaie,167,33.53
5,e-commerce,2342,3.20
6,supermarché,2060,1.12
7,carburant,2177,0.96
8,téléphonie,2128,0.85
9,pharmacie,2130,0.80


## 2. Outils SQL (lecture seule)

In [4]:
from app import tools
h = tools.get_transaction_history('CARD-00262', limit=5)
print(h['stats']); pd.DataFrame(h['transactions'])

{'n_tx': 77, 'avg_amount': 76.19, 'max_amount': 723.21, 'n_foreign': 7}


,tx_id,ts,amount,merchant_category,country
0,119765,2026-09-30T13:02:45,5.13,e-commerce,FR
1,119654,2026-09-30T09:27:00,0.95,loisirs,FR
2,118893,2026-09-29T08:35:45,14.10,restaurant,FR
3,118806,2026-09-29T06:19:15,6.97,pharmacie,FR
4,118774,2026-09-29T05:23:45,9.99,supermarché,FR


In [5]:
tools.get_card_status('CARD-99999')  # carte inconnue -> erreur explicite

{'error': 'CARD_NOT_FOUND',
 'message': "La carte CARD-99999 n'existe pas dans la base."}

## 3. Scoring via l'API de détection de fraude du projet 2

In [6]:
for tx in [110613, 111410, 117663]:
    r = tools.score_transaction(tx)
    print(tx, r.get('fraud_probability', r.get('error')), r.get('card_id'))

110613 0.9999 CARD-00262
111410 0.0 CARD-00168
117663 0.4053 CARD-00121


## 4. RAG sur les procédures internes (Chroma)

In [7]:
from app.rag import search_procedures
for p in search_procedures("l'API de scoring ne répond plus")['passages']:
    print(p['procedure_id'], '|', p['section'], '|', p['score'])

PROC-07 | Introduction | 0.239
PROC-03 | Signaux rassurants | 0.152
PROC-04 | Client injoignable | 0.087


## 5. Le serveur MCP
L'agent découvre ses outils via le protocole MCP (stdio).

In [8]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
params = StdioServerParameters(command=sys.executable, args=['-m','mcp_server.server'], cwd=str(ROOT),
    env={**os.environ, 'PYTHONPATH': str(ROOT), 'BANK_DB_PATH': str(config.BANK_DB_PATH)})
async def list_tools():
    async with stdio_client(params) as (r, w), ClientSession(r, w) as s:
        await s.initialize(); return [(t.name, t.description.splitlines()[0]) for t in (await s.list_tools()).tools]
await list_tools()

[('score_transaction',
  'Score une transaction avec le modèle de détection de fraude (API du projet 2, XGBoost).'),
 ('get_transaction_history',
  "Historique des dernières transactions d'une carte (requête SQL en lecture seule)"),
 ('get_card_status',
  "Statut d'une carte (active / blocked), titulaire, plafond et dernières actions."),
 ('block_card',
  "ACTION RISQUÉE. Demande le blocage d'une carte. La carte N'EST PAS bloquée :")]

## 6. Skills et constructeur de contexte

In [9]:
from types import SimpleNamespace
from app.skills import SKILL_REGISTRY, build_instruction
for s in SKILL_REGISTRY.values(): print(f'{s.name:18} -> {s.allowed_tools}')
ctx = SimpleNamespace(state={'active_skill': 'decision_prep', 'case': {'card_id': 'CARD-00262', 'tx_id': 110613,
    'fraud_probability': 0.9999, 'cited_procedures': ['PROC-01','PROC-02']}})
print(build_instruction(ctx)[-900:])

investigation      -> ('get_card_status', 'get_transaction_history', 'score_transaction')
procedure_lookup   -> ('search_procedures',)
decision_prep      -> ('block_card', 'get_card_status')
la procédure interne applicable (RAG sur les procédures de la banque).
- decision_prep : Proposer une action justifiée (ex. blocage) soumise à validation humaine.

Skill active : decision_prep
Instructions :
Décide selon la procédure citée :
- Si la procédure demande un blocage (score >= 0,90 ou score élevé + signaux aggravants) : appelle block_card avec card_id, tx_id, une justification (score, signaux) et le procedure_id. La carte n'est PAS bloquée tant qu'un humain n'a pas approuvé.
- Sinon (score faible, carte inconnue, API indisponible, carte déjà bloquée) : n'appelle PAS block_card, recommande l'action prévue (classer, contacter le client, escalader N2...).
Termine par la réponse finale au format demandé.
Outils autorisés : block_card, get_card_status

Dossier en cours :
- card_id : CARD-0026

## 7. Garde-fous

In [10]:
from app.guardrails import check_tool_call
state = {'active_skill': 'decision_prep'}
block = {'card_id': 'CARD-00262', 'procedure_id': 'PROC-01', 'justification': 'score 0.9999, 3 pays étrangers'}
print(check_tool_call('block_card', block, state, 'inv1'))       # refusé : aucune procédure consultée
print(check_tool_call('score_transaction', {'tx_id': 1}, state, 'inv1'))  # refusé : hors skill
state['active_skill'] = 'investigation'
print(check_tool_call('get_card_status', {'card_id': 'CARD-99999'}, state, 'inv1'))  # carte inconnue

guardrail GUARDRAIL_NO_PROCEDURE on block_card: block_card exige une procédure retrouvée via search_procedures. Procédures consultées : aucune.


guardrail GUARDRAIL_SKILL on score_transaction: L'outil score_transaction n'est pas autorisé dans la skill active (decision_prep). Active d'abord la skill 'investigation'.


guardrail CARD_NOT_FOUND on get_card_status: La carte CARD-99999 n'existe pas. Aucune action possible (PROC-06).


{'error': 'GUARDRAIL_NO_PROCEDURE', 'message': 'block_card exige une procédure retrouvée via search_procedures. Procédures consultées : aucune.', 'blocked_by_guardrail': True}
{'error': 'GUARDRAIL_SKILL', 'message': "L'outil score_transaction n'est pas autorisé dans la skill active (decision_prep). Active d'abord la skill 'investigation'.", 'blocked_by_guardrail': True}
{'error': 'CARD_NOT_FOUND', 'message': "La carte CARD-99999 n'existe pas. Aucune action possible (PROC-06).", 'blocked_by_guardrail': True}


## 8. Validation humaine (human-in-the-loop)

In [11]:
from app import db
req = tools.block_card('CARD-00262', 'Score 0.9999 sur la transaction 110613, catégorie à risque.', 'PROC-01', 110613)
print(req['message']); print('statut carte :', db.get_card_status('CARD-00262')['status'])
print(db.decide_action(req['action_id'], approve=True, reviewer='analyste N1'))
print('statut carte après approbation :', db.get_card_status('CARD-00262')['status'])

Demande de blocage enregistrée, EN ATTENTE DE VALIDATION HUMAINE. La carte n'est pas encore bloquée.
statut carte : active
{'action_id': 1, 'status': 'executed', 'card_id': 'CARD-00262'}
statut carte après approbation : blocked


## 9. Évaluation : agent vs assistant à simple prompt
Générée par `bash scripts/run_eval.sh` (25 scénarios, LLM juge).

In [12]:
from IPython.display import Markdown
p = ROOT/'docs'/'eval_results.md'
Markdown(p.read_text() if p.exists() else '_Lancer `bash scripts/run_eval.sh` pour produire les résultats._')

_Lancer `bash scripts/run_eval.sh` pour produire les résultats._